# 🏛️ Bangladesh Law - Custom Model Server
## Connect Your Colab Model to Bangladesh Law App

This notebook exposes your custom model as an API endpoint that the Bangladesh Law MERN app can call.

### Steps:
1. Load your trained model below
2. Run all cells
3. Copy the ngrok public URL
4. Paste it in the Bangladesh Law app → Model Config panel
5. Click 'Connect & Switch'

In [ ]:
# Install dependencies
!pip install flask flask-cors pyngrok -q

In [ ]:
# ============================================================
# CELL 1: Load YOUR custom model here
# ============================================================
# Example: Hugging Face Transformers
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# tokenizer = AutoTokenizer.from_pretrained('your-model-path')
# model = AutoModelForSeq2SeqLM.from_pretrained('your-model-path')

# Example: Sentence Transformers + custom retriever
# from sentence_transformers import SentenceTransformer
# encoder = SentenceTransformer('your-encoder')

# Example: LangChain + FAISS
# from langchain.vectorstores import FAISS
# from langchain.chains import RetrievalQA
# qa_chain = RetrievalQA.from_chain_type(...)

# ==============================
# REPLACE THIS WITH YOUR MODEL:
# ==============================
def your_model_predict(question, language, conversation_history=None):
    """
    Replace this function with your actual model inference.
    
    Args:
        question (str): User's question (English or Bengali)
        language (str): 'en' or 'bn'
        conversation_history (list): Previous Q&A pairs
    
    Returns:
        str: Model's answer
    """
    # TODO: Replace with your model inference
    # Example:
    # inputs = tokenizer(question, return_tensors='pt', max_length=512)
    # outputs = model.generate(**inputs, max_new_tokens=300)
    # return tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Placeholder response for testing:
    if language == 'bn':
        return f"আপনার প্রশ্ন: '{question}' - এটি আপনার কাস্টম মডেলের উত্তর হবে। এখানে আপনার মডেলের inference কোড যোগ করুন।"
    return f"Your question: '{question}' - This is where your custom model's answer will appear. Replace the placeholder with your actual model inference."

print('✅ Model function defined. Replace with your actual model.')

In [ ]:
# ============================================================
# CELL 2: Flask API Server
# ============================================================
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import re

app = Flask(__name__)
CORS(app)  # Allow cross-origin requests from your MERN app

def detect_language(text):
    """Detect if text is Bengali or English"""
    bengali_pattern = re.compile(r'[\u0980-\u09FF]')
    return 'bn' if bengali_pattern.search(text) else 'en'

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'custom', 'server': 'Bangladesh Law Colab'})

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        
        question = data.get('question', '').strip()
        language = data.get('language') or detect_language(question)
        conversation_history = data.get('conversation_history', [])
        
        if not question:
            return jsonify({'error': 'Question is required'}), 400
        
        print(f'[{language.upper()}] Q: {question[:80]}...' if len(question) > 80 else f'[{language.upper()}] Q: {question}')
        
        # Call your model
        answer = your_model_predict(question, language, conversation_history)
        
        print(f'A: {str(answer)[:80]}...' if len(str(answer)) > 80 else f'A: {answer}')
        
        return jsonify({
            'answer': answer,
            'language': language,
            'model': 'custom'
        })
    
    except Exception as e:
        print(f'Error: {e}')
        return jsonify({'error': str(e)}), 500

print('✅ Flask app configured')

In [ ]:
# ============================================================
# CELL 3: Start Server with ngrok tunnel
# ============================================================
from pyngrok import ngrok
import os

# Optional: Set your ngrok auth token for stable URLs
# Get free token at: https://dashboard.ngrok.com
NGROK_TOKEN = ''  # paste your token here if you have one
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start Flask in background thread
port = 5001
flask_thread = threading.Thread(target=lambda: app.run(port=port, debug=False, use_reloader=False))
flask_thread.daemon = True
flask_thread.start()

# Create ngrok tunnel
public_url = ngrok.connect(port).public_url

print('=' * 60)
print('🏛️ Bangladesh Law Custom Model Server is LIVE!')
print('=' * 60)
print(f'\n🔗 PUBLIC URL (copy this):')
print(f'\n   {public_url}\n')
print('📋 Steps to connect:')
print('   1. Copy the URL above')
print('   2. Open your Bangladesh Law app')
print('   3. Click the model status bar (top of chat)')
print('   4. Paste URL in the Colab URL field')
print('   5. Click "Connect & Switch"')
print('\n✅ Health check:', f'{public_url}/health')
print('=' * 60)

In [ ]:
# ============================================================
# CELL 4: Test your model locally before connecting
# ============================================================
import requests

# Test English
r = requests.post(f'http://localhost:{port}/predict', json={
    'question': 'What are fundamental rights in Bangladesh Constitution?',
    'language': 'en'
})
print('English test:', r.json()['answer'][:200])

# Test Bengali
r2 = requests.post(f'http://localhost:{port}/predict', json={
    'question': 'বাংলাদেশের সংবিধানে মৌলিক অধিকার কী?',
    'language': 'bn'
})
print('Bengali test:', r2.json()['answer'][:200])